# Elastic Late Chunking Demo

## Create Elastic Serverless Environment

In [ ]:
%%bash
echo "--- Initializing ---"
terraform -chdir=terraform init -upgrade -input=false > /dev/null && echo "Done."
echo "--- Applying Changes ---"
terraform -chdir=terraform apply -auto-approve > /dev/null && echo "Done."

echo "--- Exporting Environment Variables ---"
cat > .env << EOF
ELASTIC_CLOUD_API_KEY=$(terraform -chdir=terraform output -raw elastic_cloud_api_key)
ELASTIC_CLOUD_ID=$(terraform -chdir=terraform output -raw elastic_cloud_id)
JINA_API_KEY=$(terraform -chdir=terraform output -raw jina_api_key)
EOF
echo "Done."

## Create Inference Endpoints

In [ ]:
from dotenv import load_dotenv
from elasticsearch import Elasticsearch
import os

load_dotenv(override=True)

es = Elasticsearch(
    cloud_id=os.environ["ELASTIC_CLOUD_ID"],
    api_key=os.environ["ELASTIC_CLOUD_API_KEY"]
)
print(f'Client connected: {es.ping()}')

es.options(ignore_status=[404]).inference.delete(inference_id="jina-embeddings-v3-standard")
es.inference.put(
    task_type="text_embedding",
    inference_id="jina-embeddings-v3-standard",
    body={
        "service": "jinaai",
        "service_settings": {
            "api_key": os.environ["JINA_API_KEY"],
            "model_id": "jina-embeddings-v3",
            
        },
        "task_settings": {
            "late_chunking": False,
            "input_type": "search"
        }
    }
)
print("Standard inference endpoint created")
es.options(ignore_status=[404]).inference.delete(inference_id="jina-embeddings-v3-late")
es.inference.put(
    task_type="text_embedding",
    inference_id="jina-embeddings-v3-late",
    body={
        "service": "jinaai",
        "service_settings": {
            "api_key": os.environ["JINA_API_KEY"],
            "model_id": "jina-embeddings-v3",
            
        },
        "task_settings": {
            "late_chunking": True, 
            "input_type": "search"
        }
    }
)
print("Late inference endpoint created")

## Load Indices

In [ ]:
from late_chunking import faq, filing

faq.create_index(es)
faq.ingest(es, "assets/corpus_a_faq.json")
filing.create_index(es)
filing.ingest(es, "assets/corpus_b_filing.json")

## Retrieval & Measurement

This cell evaluates both embedding strategies against a set of hand-labeled relevance judgments stored in `assets/judgments.json`. Each judgment entry contains a query, the expected relevant chunk ID(s), and a relevance rating. Elasticsearch's `_rank_eval` API scores each retrieval approach at `k=5`.

**Metrics used:**

- **MRR (Mean Reciprocal Rank @ 5)** — For each query, takes the reciprocal of the rank at which the first relevant result appears (1/1, 1/2, 1/3, …, 0 if not in top 5), then averages across all queries. A score of 1.0 means the correct chunk was always ranked first; 0.5 means it was typically second. MRR is sensitive to the position of the single best result.
- **NDCG (Normalized Discounted Cumulative Gain @ 5)** — A graded ranking metric that rewards placing more-relevant results higher in the list, with a logarithmic discount for lower positions. Scores are normalized against the ideal ranking (IDCG), so 1.0 is a perfect ranking. NDCG captures the full shape of the top-5 list, not just the first hit.

**Query strategy difference:**

Standard indices are queried with Elasticsearch's `semantic` query, which embeds the query string through the standard inference endpoint and performs cosine similarity search. Late indices cannot use the `semantic` query wrapper in the same way — instead, the query string is explicitly embedded via the late-chunking inference endpoint, and the resulting vector is passed directly to a `knn` query. This mirrors the asymmetry in how each index was populated.

---

### Results

**Corpus A — Product FAQ**

```
MRR:   Standard 1.0000  →  Late 0.5857   Δ −0.4143
NDCG:  Standard 0.9868  →  Late 0.6161   Δ −0.3707
```

Standard embedding is essentially perfect on the FAQ corpus. Every query returns the correct chunk at rank 1. Late chunking degrades both metrics by roughly 40 points — a severe regression.

The reason is structural. FAQ entries are atomic, self-contained Q&A pairs with no meaningful relationship to their neighbors in the document. When late chunking embeds the full FAQ document as a single sequence, the attention mechanism allows adjacent — but semantically unrelated — questions and answers to influence each chunk's vector. A question about password resets bleeds into the embedding of a question about return policies. This inter-chunk noise dilutes the clean, tightly scoped signal that makes independent per-chunk embedding so effective here. Standard embedding treats each FAQ as its own document; that isolation is a feature, not a limitation.

**Corpus B — SEC 10-K Filings**

```
MRR:   Standard 0.4028  →  Late 0.7917   Δ +0.3889
NDCG:  Standard 0.5539  →  Late 0.8462   Δ +0.2923
```

The pattern reverses completely. Standard embedding performs poorly — an MRR of 0.40 means the correct chunk is frequently not even the top result. Late chunking nearly doubles MRR and pushes NDCG above 0.84.

10-K filings are the opposite of FAQs: dense, cross-referential prose where individual sections are not self-contained. A passage like "As discussed in the liquidity section, the allowance for credit losses declined" is nearly uninterpretable in isolation — "the allowance" and "the liquidity section" require surrounding context to resolve. Defined terms are introduced once and referenced throughout; pronouns like "the Company" or "our" carry meaning only in relation to the document they appear in. When each section is embedded independently, the model has no access to this context and produces vectors that reflect the surface tokens rather than the meaning. Retrieval quality suffers accordingly.

Late chunking fixes this by giving the model the full document before pooling per-chunk vectors. Each section's embedding is shaped by the same attention over the entire filing, so cross-references, defined terms, and pronouns are all resolved before the vector is produced. The retrieval improvement is substantial and consistent across both MRR and NDCG.

In [ ]:
import json
from late_chunking import faq, filing

with open("assets/judgments.json") as f:
    judgments = json.load(f)

METRICS = {
    "MRR":  {"mean_reciprocal_rank": {"k": 5}},
    "NDCG": {"dcg":                  {"k": 5, "normalize": True}},
}

def build_std_requests(index, queries):
    return [
        {
            "id": str(i),
            "request": {
                "query": {"semantic": {"field": "embedding", "query": item['query']}},
                "size": 5
            },
            "ratings": [{"_index": index, "_id": r['chunk_id'], "rating": r['rating']} for r in item['ratings']]
        }
        for i, item in enumerate(queries)
    ]

def build_late_requests(es, index, queries):
    requests = []
    for i, item in enumerate(queries):
        vec = es.inference.inference(input=[item['query']], inference_id="jina-embeddings-v3-late")
        requests.append({
            "id": str(i),
            "request": {
                "knn": {"field": "embedding", "query_vector": vec['text_embedding'][0]['embedding'], "k": 5, "num_candidates": 50},
                "size": 5
            },
            "ratings": [{"_index": index, "_id": r['chunk_id'], "rating": r['rating']} for r in item['ratings']]
        })
    return requests

def eval_metrics(es, index, requests):
    return {
        name: es.rank_eval(index=index, requests=requests, metric=metric)['metric_score']
        for name, metric in METRICS.items()
    }

def report(label, es, std_index, late_index, queries):
    print(f"=== {label} ===\n")
    std_req  = build_std_requests(std_index, queries)
    late_req = build_late_requests(es, late_index, queries)
    std_scores  = eval_metrics(es, std_index,  std_req)
    late_scores = eval_metrics(es, late_index, late_req)
    print(f"{'Metric':<8} {'Standard':>10} {'Late':>10} {'Delta':>10}")
    print("-" * 40)
    for name in METRICS:
        s, l = std_scores[name], late_scores[name]
        print(f"{name:<8} {s:>10.4f} {l:>10.4f} {l - s:>+10.4f}")
    print()

report("Corpus A: Product FAQ", es, faq.FAQ_STANDARD_INDEX_NAME,         faq.FAQ_LATE_INDEX_NAME,             judgments['corpus_a'])
report("Corpus B: SEC 10-K Filings", es, filing.FILING_STANDARD_INDEX_NAME,   filing.FILING_LATE_INDEX_NAME,       judgments['corpus_b'])

## Summary & Conclusions

This experiment tested a single, precise hypothesis: **is late chunking a universal upgrade to dense retrieval, or does its benefit depend on the structure of the content being indexed?**

The results are unambiguous — it depends entirely on the content.

### What was observed

| Corpus | Content type | Standard MRR | Late MRR | Winner |
|---|---|---|---|---|
| A — Product FAQ | Self-contained Q&A pairs | **1.00** | 0.59 | Standard by 0.41 |
| B — SEC 10-K Filings | Cross-referential prose | 0.40 | **0.79** | Late by 0.39 |

The magnitude of the effect is symmetric and large in both directions. This is not a subtle difference — each strategy is roughly twice as good as the other on the corpus it suits.

### Why late chunking helps some content and hurts other content

Late chunking works by embedding a full document as a single token sequence before pooling per-chunk vectors. This means each chunk's representation is influenced by every other chunk in the document via the transformer's attention mechanism.

- **When chunks are semantically independent** (FAQ entries, product catalog items, news articles), this cross-chunk attention introduces noise. Adjacent chunks are unrelated, so their influence degrades the precision of each chunk's vector. Standard embedding — treating each chunk as its own document — is better because isolation is the right model.

- **When chunks depend on surrounding context** (legal filings, technical manuals, narrative prose, academic papers), isolated embedding is lossy. Defined terms, pronoun references, section cross-references, and running context all require the model to have seen the surrounding text. Late chunking preserves that context; standard embedding discards it.

### Decision rule for practitioners

The choice between strategies reduces to a single question: **does a chunk make complete sense on its own?**

| If chunks are… | Use |
|---|---|
| Self-contained (each chunk is independently meaningful) | Standard per-chunk embedding |
| Context-dependent (cross-references, pronouns, defined terms, running narrative) | Late chunking |

Applying late chunking to a self-contained corpus like a FAQ or product catalog is not just unhelpful — it actively harms retrieval quality. The correct strategy is to match the embedding approach to the semantic structure of the content.

### Broader implications

This result matters for RAG system design. The common advice to "just use late chunking" or "late chunking is always better" is not supported by evidence. Production systems that mix content types (e.g., a knowledge base containing both policy FAQs and technical reference documentation) may need separate indices with different embedding strategies routed at query time based on content type. A one-size-fits-all embedding pipeline leaves significant retrieval quality on the table in one direction or the other.

## Destroy Environment

In [ ]:
%%bash
terraform -chdir=terraform destroy -auto-approve > /dev/null && echo "Done."
rm -f .env